In [9]:
"""
8. 텔롭 포지션 — 인스턴스별 대표 bbox 집계
  Input:  3_ocr_results + 5_telop_segments
  Output: 8_telop_position/{channel}/{video_name}.json

  각 인스턴스에 text, start_sec, end_sec + x, y, w, h (좌상단 기준) 포함
  pixel 좌표 (720x1280) + grid 좌표 (72x128, 10px 단위)
"""
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
OCR_DIR = "./data/3_ocr_results"
SEG_DIR = "./data/5_telop_segments"
OUT_DIR = "./data/8_telop_position"
FPS = 10
NUM_WORKERS = 32

FRAME_W = 720
FRAME_H = 1280
GRID_W = 72   # 720 / 10
GRID_H = 128  # 1280 / 10
CELL_SIZE = 10


def _loads(x):
    return json.loads(x) if isinstance(x, str) else x


def process_one(ocr_pq: str, seg_pq: str, out_json: str, video_name: str):
    df_ocr = pd.read_parquet(ocr_pq).set_index("frame_num")
    df_seg = pd.read_parquet(seg_pq).set_index("frame_num")

    tracks = defaultdict(lambda: {
        "frames": set(),
        "text_scores": defaultdict(float),
        "xywhas": [],
    })

    for fnum, ocr_row in df_ocr.iterrows():
        if fnum not in df_seg.index:
            continue

        texts = _loads(ocr_row["ocr_texts"])
        scores = _loads(ocr_row["ocr_scores"])
        xywhas = _loads(ocr_row["ocr_xywha"])
        tids = _loads(df_seg.loc[fnum, "track_id"])

        for text, score, xywha, tid in zip(texts, scores, xywhas, tids):
            if tid < 0:
                continue
            tracks[tid]["frames"].add(int(fnum))
            tracks[tid]["text_scores"][text] += float(score)
            tracks[tid]["xywhas"].append(xywha)  # [cx, cy, w, h, angle]

    instances = []
    for tid, info in tracks.items():
        frames = sorted(info["frames"])
        best_text = max(info["text_scores"].items(), key=lambda kv: kv[1])[0]

        # 대표 bbox: 프레임별 xywha의 median (angle 제외)
        arr = np.array(info["xywhas"])  # (N, 5) = cx, cy, w, h, angle
        cx_med = float(np.median(arr[:, 0]))
        cy_med = float(np.median(arr[:, 1]))
        w_med = float(np.median(arr[:, 2]))
        h_med = float(np.median(arr[:, 3]))

        # center → top-left (pixel)
        px_x = cx_med - w_med / 2
        px_y = cy_med - h_med / 2

        # pixel → grid 좌표 (72x128)
        gx = px_x / CELL_SIZE
        gy = px_y / CELL_SIZE
        gw = w_med / CELL_SIZE
        gh = h_med / CELL_SIZE

        # clamp to grid bounds
        gx = max(0, min(GRID_W - 1, gx))
        gy = max(0, min(GRID_H - 1, gy))
        gw = max(0.5, min(GRID_W - gx, gw))
        gh = max(0.5, min(GRID_H - gy, gh))

        instances.append({
            "text": best_text,
            "start_sec": round(frames[0] / FPS, 3),
            "end_sec": round((frames[-1] + 1) / FPS, 3),
            # pixel 좌표 (720x1280), 좌상단
            "px_x": round(px_x, 1),
            "px_y": round(px_y, 1),
            "px_w": round(w_med, 1),
            "px_h": round(h_med, 1),
            # grid 좌표 (72x128), 좌상단
            "grid_x": round(gx, 2),
            "grid_y": round(gy, 2),
            "grid_w": round(gw, 2),
            "grid_h": round(gh, 2),
        })

    instances.sort(key=lambda x: (x["start_sec"], x["end_sec"]))

    result = {
        "video": video_name,
        "frame_size": [FRAME_W, FRAME_H],
        "grid_size": [GRID_W, GRID_H],
        "instances": instances,
    }

    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


def main():
    tasks = []
    for channel in sorted(os.listdir(SEG_DIR)):
        seg_ch = os.path.join(SEG_DIR, channel)
        if not os.path.isdir(seg_ch):
            continue
        for fname in sorted(os.listdir(seg_ch)):
            if not fname.endswith(".parquet"):
                continue
            vn = fname[:-8]
            vn_clean = vn.rsplit("__", 1)[0]
            seg_pq = os.path.join(seg_ch, fname)
            ocr_pq = os.path.join(OCR_DIR, channel, fname)
            out_json = os.path.join(OUT_DIR, channel, vn + ".json")
            if not os.path.exists(ocr_pq) or os.path.exists(out_json):
                continue
            tasks.append((ocr_pq, seg_pq, out_json, vn_clean))

    print(f"🎯 {len(tasks):,}개 처리 예정 (이미 존재하는 파일은 skip)")

    done = 0
    errors = 0
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(process_one, *t): t for t in tasks}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="처리"):
            try:
                fut.result()
                done += 1
            except Exception as e:
                errors += 1
                task = futures[fut]
                tqdm.write(f"  ⚠️ {task[3]}: {e}")

    print(f"\n✅ 완료: {done:,}개  |  오류: {errors:,}개")


if __name__ == "__main__":
    main()

🎯 66,400개 처리 예정 (이미 존재하는 파일은 skip)


처리: 100%|██████████| 66400/66400 [01:39<00:00, 669.08it/s] 



✅ 완료: 66,400개  |  오류: 0개


In [10]:
"""
8_telop_position 데이터 분포 분석
- grid 좌표(x, y, w, h) 분포
- 채널별 위치 정형화 정도
- 동시 텔롭 수별 위치 패턴
"""
import os
import json
import glob
import numpy as np
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

POS_DIR = "./data/8_telop_position"
NUM_WORKERS = 32
GRID_W = 72
GRID_H = 128


def analyze_one(json_path: str) -> dict:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    channel = json_path.split(os.sep)[-2]

    if not instances:
        return {"channel": channel, "n": 0, "bboxes": [], "path": json_path}

    bboxes = []
    for inst in instances:
        bboxes.append({
            "x": inst["grid_x"],
            "y": inst["grid_y"],
            "w": inst["grid_w"],
            "h": inst["grid_h"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
        })

    return {"channel": channel, "n": len(instances), "bboxes": bboxes, "path": json_path}


def main():
    json_paths = sorted(glob.glob(os.path.join(POS_DIR, "**", "*.json"), recursive=True))
    print(f"📁 분석 대상: {len(json_paths):,}개 영상")

    results = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(analyze_one, p): p for p in json_paths}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="분석"):
            results.append(fut.result())

    # ── 전체 bbox 수집 ──
    all_x, all_y, all_w, all_h, all_tlen = [], [], [], [], []
    channel_positions = defaultdict(lambda: {"x": [], "y": []})

    for r in results:
        for b in r["bboxes"]:
            all_x.append(b["x"])
            all_y.append(b["y"])
            all_w.append(b["w"])
            all_h.append(b["h"])
            all_tlen.append(b["text_len"])
            channel_positions[r["channel"]]["x"].append(b["x"])
            channel_positions[r["channel"]]["y"].append(b["y"])

    all_x = np.array(all_x)
    all_y = np.array(all_y)
    all_w = np.array(all_w)
    all_h = np.array(all_h)
    all_tlen = np.array(all_tlen)

    print(f"\n{'='*60}")
    print(f"📊 전체 통계 ({len(all_x):,}개 인스턴스)")
    print(f"{'='*60}")

    for name, arr, grid_max in [("x", all_x, GRID_W), ("y", all_y, GRID_H),
                                 ("w", all_w, GRID_W), ("h", all_h, GRID_H)]:
        print(f"\n  grid_{name} (범위 0~{grid_max-1}):")
        print(f"    mean: {arr.mean():.1f}  median: {np.median(arr):.1f}  "
              f"std: {arr.std():.1f}  min: {arr.min():.1f}  max: {arr.max():.1f}")

    # ── y 위치 분포 (상단/중단/하단) ──
    print(f"\n📊 y 위치 분포 (grid_h=128)")
    y_bins = [(0, 25, "상단 (0~25)"), (25, 50, "중상단 (25~50)"),
              (50, 75, "중단 (50~75)"), (75, 100, "중하단 (75~100)"),
              (100, 128, "하단 (100~128)")]
    for lo, hi, label in y_bins:
        cnt = int(((all_y >= lo) & (all_y < hi)).sum())
        print(f"    {label:<20} {cnt:>8} ({cnt/len(all_y)*100:>5.1f}%)")

    # ── x 위치 분포 (좌/중/우) ──
    print(f"\n📊 x 위치 분포 (grid_w=72)")
    x_bins = [(0, 18, "좌측 (0~18)"), (18, 36, "중좌 (18~36)"),
              (36, 54, "중우 (36~54)"), (54, 72, "우측 (54~72)")]
    for lo, hi, label in x_bins:
        cnt = int(((all_x >= lo) & (all_x < hi)).sum())
        print(f"    {label:<20} {cnt:>8} ({cnt/len(all_x)*100:>5.1f}%)")

    # ── 텍스트 길이 vs w 상관관계 ──
    corr = np.corrcoef(all_tlen, all_w)[0, 1]
    print(f"\n📊 텍스트 길이 vs grid_w 상관계수: {corr:.3f}")

    # ── 채널별 위치 분산 ──
    print(f"\n📊 채널별 y 위치 표준편차 (정형화 정도)")
    ch_y_stds = []
    for ch, pos in channel_positions.items():
        if len(pos["y"]) >= 10:
            ch_y_stds.append((ch, np.std(pos["y"]), len(pos["y"])))

    ch_y_stds.sort(key=lambda x: x[1])
    print(f"\n  가장 정형화된 채널 (y std 낮음):")
    for ch, std, n in ch_y_stds[:10]:
        print(f"    std={std:>5.1f}  (n={n:>5})  {ch}")

    print(f"\n  가장 다양한 채널 (y std 높음):")
    for ch, std, n in ch_y_stds[-10:]:
        print(f"    std={std:>5.1f}  (n={n:>5})  {ch}")

    all_stds = [s for _, s, _ in ch_y_stds]
    print(f"\n  전체 채널 y std — mean: {np.mean(all_stds):.1f}  median: {np.median(all_stds):.1f}")

    # ── w, h 분포 ──
    print(f"\n📊 bbox 크기 분포")
    print(f"  grid_w 구간:")
    for lo, hi in [(0, 5), (5, 10), (10, 20), (20, 40), (40, 72)]:
        cnt = int(((all_w >= lo) & (all_w < hi)).sum())
        print(f"    {lo:>2}~{hi:<3} {cnt:>8} ({cnt/len(all_w)*100:>5.1f}%)")

    print(f"  grid_h 구간:")
    for lo, hi in [(0, 2), (2, 4), (4, 8), (8, 16), (16, 128)]:
        cnt = int(((all_h >= lo) & (all_h < hi)).sum())
        print(f"    {lo:>2}~{hi:<3} {cnt:>8} ({cnt/len(all_h)*100:>5.1f}%)")

    # ── 음수 좌표 체크 ──
    neg_x = (all_x < 0).sum()
    neg_y = (all_y < 0).sum()
    if neg_x > 0 or neg_y > 0:
        print(f"\n  ⚠️ 음수 좌표: x={neg_x}, y={neg_y}")


if __name__ == "__main__":
    main()

📁 분석 대상: 66,392개 영상


분석: 100%|██████████| 66392/66392 [00:06<00:00, 10737.65it/s]



📊 전체 통계 (3,740,728개 인스턴스)

  grid_x (범위 0~71):
    mean: 23.4  median: 19.4  std: 18.6  min: 0.0  max: 71.0

  grid_y (범위 0~127):
    mean: 61.5  median: 66.3  std: 32.5  min: 0.0  max: 127.0

  grid_w (범위 0~71):
    mean: 19.5  median: 14.0  std: 16.4  min: 0.5  max: 72.0

  grid_h (범위 0~127):
    mean: 4.8  median: 3.9  std: 3.4  min: 0.5  max: 79.5

📊 y 위치 분포 (grid_h=128)
    상단 (0~25)              682060 ( 18.2%)
    중상단 (25~50)            780104 ( 20.9%)
    중단 (50~75)             698257 ( 18.7%)
    중하단 (75~100)          1155267 ( 30.9%)
    하단 (100~128)           425040 ( 11.4%)

📊 x 위치 분포 (grid_w=72)
    좌측 (0~18)             1760518 ( 47.1%)
    중좌 (18~36)            1119239 ( 29.9%)
    중우 (36~54)             457804 ( 12.2%)
    우측 (54~72)             403167 ( 10.8%)

📊 텍스트 길이 vs grid_w 상관계수: 0.574

📊 채널별 y 위치 표준편차 (정형화 정도)

  가장 정형화된 채널 (y std 낮음):
    std= 10.8  (n=35170)  김준표
    std= 12.0  (n= 9456)  EBSDocumentary (EBS 다큐)
    std= 12.1  (n=44988)  임다TV
    std= 12.6  (